In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/dharmrv/tinyshakespeare/input.txt


In [2]:
import torch
import torch.nn as nn
from torch.nn import functional as F

In [3]:
text = open('/kaggle/input/datasets/dharmrv/tinyshakespeare/input.txt' ,'r').read()
text[:1000]

"First Citizen:\nBefore we proceed any further, hear me speak.\n\nAll:\nSpeak, speak.\n\nFirst Citizen:\nYou are all resolved rather to die than to famish?\n\nAll:\nResolved. resolved.\n\nFirst Citizen:\nFirst, you know Caius Marcius is chief enemy to the people.\n\nAll:\nWe know't, we know't.\n\nFirst Citizen:\nLet us kill him, and we'll have corn at our own price.\nIs't a verdict?\n\nAll:\nNo more talking on't; let it be done: away, away!\n\nSecond Citizen:\nOne word, good citizens.\n\nFirst Citizen:\nWe are accounted poor citizens, the patricians good.\nWhat authority surfeits on would relieve us: if they\nwould yield us but the superfluity, while it were\nwholesome, we might guess they relieved us humanely;\nbut they think we are too dear: the leanness that\nafflicts us, the object of our misery, is as an\ninventory to particularise their abundance; our\nsufferance is a gain to them Let us revenge this with\nour pikes, ere we become rakes: for the gods know I\nspeak this in hunger 

In [4]:
chars =sorted(list(set(text)))
vocab_size =len(chars)
print(''.join(chars))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [5]:
stoi = {s:i for i,s in enumerate(chars)}
itos = {i:s for s,i in stoi.items()}
encode = lambda s: [stoi[c] for c in s] 
decode = lambda l: ''.join([itos[c] for c in l])

print(encode("hi there"))
print(decode(encode("hi there")))

[46, 47, 1, 58, 46, 43, 56, 43]
hi there


prepare data from text  (encoding full data)

In [6]:
data = torch.tensor(encode(text) , dtype = torch.long)
data.shape , data.dtype
# data[:]

(torch.Size([1115394]), torch.int64)

In [7]:
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]
train_data.shape

torch.Size([1003854])

In [8]:
block_size =8
train_data[:block_size+1]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [9]:
x= train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
    context = x[:t+1]
    target  =y[t]
    print(f'input:{context} -> output : {target} ')

input:tensor([18]) -> output : 47 
input:tensor([18, 47]) -> output : 56 
input:tensor([18, 47, 56]) -> output : 57 
input:tensor([18, 47, 56, 57]) -> output : 58 
input:tensor([18, 47, 56, 57, 58]) -> output : 1 
input:tensor([18, 47, 56, 57, 58,  1]) -> output : 15 
input:tensor([18, 47, 56, 57, 58,  1, 15]) -> output : 47 
input:tensor([18, 47, 56, 57, 58,  1, 15, 47]) -> output : 58 


In [10]:
torch.manual_seed(1337)
n_embd=384
batch_size =64
block_size =256
n_head =6
n_layer =6
dropout = 0.2
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

def get_batch(split):
    data = train_data if split =='train' else val_data
    ix = torch.randint(len(data) - block_size , (batch_size,))
    x= torch.stack([data[i:i+block_size] for i in ix])
    y= torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x.to(device), y.to(device)

xb,yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print("output:")
print(yb.shape)
print(yb)

for b in range(batch_size):
    for t in range(block_size):
        context  = xb[b, :t+1]
        target  = yb[b,t]

        # print(f'when input is {context.tolist()} the target is  {target}')

Using device: cuda
inputs:
torch.Size([64, 256])
tensor([[ 0, 26, 53,  ..., 56, 43, 47],
        [60, 43, 56,  ..., 56,  1, 41],
        [26, 21, 33,  ..., 26, 21, 13],
        ...,
        [ 5, 57,  1,  ...,  1, 35, 47],
        [56, 53, 53,  ..., 59, 50, 42],
        [42, 47, 56,  ..., 39, 56,  1]], device='cuda:0')
output:
torch.Size([64, 256])
tensor([[26, 53, 58,  ..., 43, 47, 45],
        [43, 56,  1,  ...,  1, 41, 53],
        [21, 33, 31,  ..., 21, 13, 10],
        ...,
        [57,  1, 52,  ..., 35, 47, 50],
        [53, 53, 58,  ..., 50, 42,  1],
        [47, 56, 43,  ..., 56,  1, 51]], device='cuda:0')


In [ ]:





class Head(nn.Module):
   
    def __init__(self,head_size):
        super().__init__()
        self.key = nn.Linear(n_embd , head_size, bias = False)
        self.query = nn.Linear(n_embd , head_size, bias = False)
        self.value = nn.Linear(n_embd , head_size, bias = False)
        self.register_buffer('tril' , torch.tril(torch.ones(block_size,block_size)))
        self.dropout =nn.Dropout(dropout)

    def forward(self,x):
        B,T,C = x.shape
        k = self.key(x)
        q = self.query(x)

        wei = q @k.transpose(-2,-1) * (C**-0.5)
        wei= wei.masked_fill(self.tril[:T,:T]==0 , float('-inf'))
        wei =F.softmax(wei , dim =-1)
        wei = self.dropout(wei)
        
        v = self.value(x)
        out = wei @ v
        return out 


class MultiHeadAttention(nn.Module):
    def __init__(self ,num_heads,head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(num_heads* head_size , n_embd)
        self.dropout =nn.Dropout(dropout)

    def forward(self,x):
        out = torch.cat([h(x) for h in self.heads] , dim =-1)
        out = self.proj(out)
        return out
        

class FeedForward(nn.Module):
    def __init__(self,n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd,4 * n_embd),
            nn.ReLU(),
            nn.Linear(4*n_embd,n_embd),
            nn.Dropout(dropout),
            
        )
    def forward(self,x):
        return self.net(x)
        
class Block(nn.Module):
    def __init__(self,n_embd , n_head):
        super().__init__()
        head_size =n_embd//n_head
        self.sa =MultiHeadAttention(n_head , head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self,x):
        x= x +self.sa(self.ln1(x))
        x =x + self.ffwd(self.ln2(x))
        return x

class BigramLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size,n_embd)
        self.position_embedding_table = nn.Embedding(block_size,n_embd)
        # print(f"token_embdding :{self.token_embedding_table.weight.sum(0)}")
        self.blocks = nn.Sequential(*[Block(n_embd,n_head=n_head) for _ in range(n_layer)])
        self.ln_f =nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd,vocab_size)


    def forward(self , idx , targets =None):
        B,T = idx.shape
        token_embed  = self.token_embedding_table(idx)
        pos_embd =  self.position_embedding_table(torch.arange(T ,device=device))
        # print(f"logits shape :{logits.shape} -> logits :{logits[0]}")
        x= token_embed + pos_embd
        # x= self.sa_heads(x)
        # # print(f'from attention :{x.shape}')
        # x= self.ffd(x)
        x= self.blocks(x)
        logits= self.lm_head(x)  # (B,T, vocab_size)
        
        b,t,c = logits.shape 
        
        
        if targets == None :
            loss = None 
        else:
           logits = logits.view(b*t,c)
           targets= targets.view(b*t)
           loss = F.cross_entropy(logits,targets)
        return logits, loss

    def generate(self,idx,max_new_token):

        for _ in range(max_new_token):

            idx_cond = idx[:,-block_size:]
            
            logits,loss =self(idx_cond)
            
            logits  = logits[: ,-1,:]
        
            probs = F.softmax(logits , dim =-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx,idx_next) , dim =1)
        return idx 


# -----------------------------------------------------------------------------------------------
class BatchNorm1d:
  
  def __init__(self, dim, eps=1e-5, momentum=0.1):
    self.eps = eps
  
    self.gamma = torch.ones(dim)
    self.beta = torch.zeros(dim)

  def __call__(self, x):

    xmean = x.mean(1, keepdim=True) # batch mean
    xvar = x.var(1, keepdim=True) # batch variance

    xhat = (x - xmean) / torch.sqrt(xvar + self.eps) # normalize to unit variance
    self.out = self.gamma * xhat + self.beta

    return self.out
  
  def parameters(self):
    return [self.gamma, self.beta]




In [12]:
# print(vocab_size)
m= BigramLanguageModel()
m= m.to(device)
logits,loss  = m(xb,yb)

print(loss.item())
idx = torch.zeros((1,1), dtype=torch.long , device =device)
print(f"idx:{idx}")
# print(decode(m.generate(idx, max_new_token = 100)))
print(decode(m.generate(idx, max_new_token = 100)[0].tolist()))


4.57517147064209
idx:tensor([[0]], device='cuda:0')

uS;QfOAOfLdn:mIVNVhBA'nn3gOQVr'g.lbCWAhtXXfRfsNuNET!rZf!vhIrfKUuoolsQ:smnAbnXFbcqji,OwHSD?b.KEBTSFmd


In [13]:
 optimizer = torch.optim.AdamW(m.parameters(),lr =3e-4)

In [14]:
lossi =[]
batch_size=32 
for i in range(10000):
    xb,yb = get_batch('train')
    logits, loss = m(xb,yb)
    optimizer.zero_grad(set_to_none = True)
    loss.backward()
    optimizer.step()
    if i %1000 ==0:
        
        print(f"i:{loss.item()}")
lossi.append(loss)
print(loss.item())

i:4.5631794929504395
i:1.7880228757858276
i:1.4779338836669922
i:1.4091277122497559
i:1.2740199565887451
i:1.1891660690307617
i:1.1686323881149292
i:1.098895788192749
i:1.051040768623352
i:0.9721602201461792
0.947938084602356


In [16]:
idx = torch.zeros((1,1), dtype=torch.long , device= device)
print(f"idx:{idx}")
# print(decode(m.generate(idx, max_new_token = 100)))
print(decode(m.generate(idx, max_new_token = 300)[0].tolist()))

idx:tensor([[0]], device='cuda:0')

Whither he stay'd, I wish followers, take my request;
Forget his palace, and my fearful virgine.

YORK:
So Clifford! why the dearest think you shalt mar
With doth him death; yet go along with his man?
Did York me not an Edward.

COMINIUS:
Nay, lo do.

MENENIUS:
Come with your cousin,
As still an exp


## mathematical trick in self attention

In [121]:
torch.manual_seed(1337)
B,T,C = 4,8,32
x= torch.randn(B,T,C)
x.shape

torch.Size([4, 8, 32])

In [122]:
xbow = torch.zeros((B,T,C)) # x bag of words
for b in range(B):
    for t in range(T):
        xprev= x[b ,:t+1]
        xbow[b,t] = torch.mean(xprev,0)


# trick for mean 

wei =torch.tril(torch.ones(T,T))
wei = wei / wei.sum(1, keepdim =True)
wei 

xbow2 = wei @ x 
print(f"xbow shape :{xbow.shape} , xbow2:{xbow.shape}")
# print(xbow2[0] )
# print(xbow[0])

all_correct = torch.allclose(xbow2, xbow, atol=1e-6)
print(all_correct)

xbow shape :torch.Size([4, 8, 32]) , xbow2:torch.Size([4, 8, 32])
True


In [123]:
### cool version 
tril = torch.tril(torch.ones(T,T))
wei = torch.zeros((T,T))
wei = wei.masked_fill(tril ==0 , float('-inf'))
wei = F.softmax(wei, dim=-1)
wei
xbow3 = wei @ x
torch.allclose(xbow3,xbow , atol=1e-6)

True

In [129]:
head_size =16
key =nn.Linear(C,head_size ,bias =False)
query =nn.Linear(C,head_size ,bias =False)
value =nn.Linear(C,head_size ,bias =False)
k= key(x)
q= query(x)
v= value(x)
wei = q @ k.transpose(-2,-1)  * head_size**-0.5# (b,t,16) * (b,16,t) ->> (b,t,t)
tril = torch.tril(torch.ones(T,T))
# wei = torch.zeros((T,T))
wei = wei.masked_fill(tril ==0 , float('-inf'))
wei = F.softmax(wei, dim=-1)
wei
out  = wei @ v
out.shape

torch.Size([4, 8, 32])

In [133]:
print(k.shape , q.shape , wei.shape)

torch.Size([4, 8, 16]) torch.Size([4, 8, 16]) torch.Size([4, 8, 8])


In [131]:
print(x)

tensor([[[ 0.1808, -0.0700, -0.3596,  ..., -0.8016,  1.5236,  2.5086],
         [-0.6631, -0.2513,  1.0101,  ...,  1.5333,  1.6097, -0.4032],
         [-0.8345,  0.5978, -0.0514,  ..., -0.4370, -1.0012, -0.4094],
         ...,
         [-0.8961,  0.0662, -0.0563,  ...,  2.1382,  0.5114,  1.2191],
         [ 0.1910, -0.3425,  1.7955,  ...,  0.3699, -0.5556, -0.3983],
         [-0.5819, -0.2208,  0.0135,  ..., -1.9079, -0.5276,  1.0807]],

        [[ 0.4562, -1.0917, -0.8207,  ...,  0.0512, -0.6576, -2.5729],
         [ 0.0210,  1.0060, -1.2492,  ...,  0.7859, -1.1501,  1.3132],
         [ 2.2007, -0.2195,  0.5427,  ..., -0.6445,  1.0834, -0.7995],
         ...,
         [ 0.3091,  1.1661, -2.1821,  ...,  0.6151,  0.6763,  0.6228],
         [ 0.0943, -0.3156,  0.7850,  ..., -1.5735,  1.3876,  0.7251],
         [ 0.6455, -0.3313, -1.0390,  ...,  0.0895, -0.3748, -0.4781]],

        [[-0.6067,  1.8328,  0.2931,  ...,  1.0041,  0.8656,  0.1688],
         [-0.2352, -0.2586,  0.0131,  ...,  0

In [130]:
print(wei[0])

tensor([[-1.9086, -1.1690, -0.4297,  2.6019,  1.6042,  3.1861, -0.5716, -0.9410],
        [-1.5159, -1.0335, -0.2334, -1.3193, -0.4143,  2.2295,  2.6633, -1.6361],
        [-1.2623, -0.2898,  1.1709,  0.9509, -2.0584,  2.1396, -0.1283, -2.5153],
        [-0.3875,  0.8154,  0.6619, -1.3230, -0.1030, -0.9164,  1.1537, -0.9063],
        [ 0.4859, -1.3519, -0.5119,  1.4696, -1.9885, -2.3175, -2.5574, -0.9744],
        [-0.7390, -1.0852, -1.5380,  0.3421,  1.3788,  3.9499, -0.8816, -0.7827],
        [-0.2394, -0.8408, -0.6203, -0.8502, -1.0101, -0.8875,  1.4994,  0.5346],
        [ 0.4806, -0.2752, -0.4692, -0.5555, -1.1353,  1.0166,  0.8411,  0.1427]],
       grad_fn=<SelectBackward0>)
